# 08 · Phase 6: demo

Builds the demo's **offline gallery** and launches the demo app (`app.py`) with a temporary public link.

| Page | What it shows |
|---|---|
| Live analysis | upload a face image or video frame; the progress strip under it lights up Detect → Heatmap → Retrieve → Explain → Check, then the verdict with P(fake) and confidence, the Grad-CAM++ heatmap, every facial-region box coloured by heat with all 9 regions ranked, the evidence summary the LLM reads (every region), the evidence cards retrieved from the corpus and the grounded explanation with the cards it cited. Needs Groq. |
| Gallery · offline | test images analysed in advance by every model: works with no internet or API key, and reveals each image's real label (including one or two the detector gets wrong). |

**Notebook settings (right sidebar)**
- **Accelerator:** GPU T4 · **Internet:** On
- **Inputs:** **01_extract_faces**, **01b_extract_celebdf** (gallery images), **02_train** (the copy with
  EfficientNet-B0). To reuse an earlier gallery instead of building a new one, also attach this notebook's
  saved output and set `BUILD_GALLERY = False`
- **Secrets:** `GROQ_API_KEY`

**Produces:** `/kaggle/working/outputs/demo/gallery/` (face, heatmap and region images plus `index.json` with
every model's answer). Building it takes about 15 minutes (Qwen's answers are spaced 60 s apart) and a few
thousand Groq tokens per model.

**Running the demo on your own computer (exhibition):**
1. Download `outputs/demo/gallery/` from this notebook's output into the project's `outputs/demo/gallery/`.
2. Download EfficientNet-B0's `best.pt` (02_train's output) to `outputs/checkpoints/best.pt`.
3. Set `GROQ_API_KEY` and run `python app.py`, then open http://localhost:7860.

Without internet the Live page cannot explain, but the Gallery page still works.

In [ ]:
# --- Project code -------------------------------------------------------------------------
# Always clones the latest code. It lives in /tmp, OUTSIDE /kaggle/working, so it is never saved into this
# notebook's output (otherwise later notebooks would pick up a stale copy). For a private repo add a
# GITHUB_TOKEN secret. Fallback: a zip of the project uploaded as a Kaggle *dataset*.
import os, sys, glob, shutil, subprocess

PROJECT = "/tmp/dfdc-project"
REPO, BRANCH = "https://github.com/Amay-M-Nair/DFDC.git", "dev"

if not os.path.exists(PROJECT):
    token = ""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO])  # don't keep the token
    else:
        found = [p for d in range(5) for p in glob.glob("/kaggle/input/" + "*/" * d + "src/config.py")]
        if not found:
            raise RuntimeError("git clone failed: add a GITHUB_TOKEN secret or attach the project zip as a dataset")
        print("WARNING: git clone failed; using the attached copy at", os.path.dirname(os.path.dirname(found[0])))
        shutil.copytree(os.path.dirname(os.path.dirname(found[0])), PROJECT,
                        ignore=shutil.ignore_patterns("data", "outputs", "__pycache__"))

shutil.rmtree("/kaggle/working/project", ignore_errors=True)  # remove stale copies saved by older notebook versions
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
!pip install -q grad-cam openai "gradio>=6.0"
# MediaPipe places the demo's atomic regions; --no-deps stops it installing a second OpenCV (opencv-contrib)
!pip install -q --no-deps "mediapipe>=1.2"
!git -C {PROJECT} log -1 --format="code version: %h  %s  (%cd)" --date=short 2>/dev/null || echo "code version: attached copy (no git)"

In [ ]:
import json, os, shutil
from pathlib import Path
import pandas as pd
from IPython.display import display

from src.kaggle_utils import setup, find, find_datasets, checkpoint_roots, setup_llm
setup(data=False)                             # outputs go to /kaggle/working/outputs (before any other src import)

from src.classifier.compare import find_checkpoints
CKPTS = find_checkpoints(checkpoint_roots())
assert "efficientnet_b0" in CKPTS, "EfficientNet-B0 not found: attach the 02_train copy that trained it"
CKPT = str(CKPTS["efficientnet_b0"])
print("detector:", CKPT)
DATASETS = {k: v for k, v in find_datasets().items() if k in ("FF++", "CelebDF")}
print("datasets:", DATASETS)
setup_llm("groq", "openai/gpt-oss-120b")

from src.demo.pipeline import GALLERY_DIR, MODELS, DemoPipeline, build_gallery, load_gallery, pick_gallery

## Settings

In [ ]:
BUILD_GALLERY = True       # False = copy the gallery from an attached earlier output of this notebook
MODELS_TO_STORE = list(MODELS)                # every model in the demo's dropdown
MIN_INTERVAL = {"qwen/qwen3.8-27b": 60}       # Groq caps Qwen at 1K output tokens per minute
MISTAKES = 2               # images the detector gets wrong (a false alarm and a missed fake, if both exist)
SEED = 0

## Gallery

In [ ]:
pipe = DemoPipeline(CKPT)
if BUILD_GALLERY:
    picks = pick_gallery(DATASETS, pipe.explainer, mistakes=MISTAKES, seed=SEED)
    display(pd.DataFrame(picks)[["dataset", "manipulation", "label", "file"]])
    print("saved", build_gallery(pipe, picks, MODELS_TO_STORE, min_interval=MIN_INTERVAL))
else:
    index = find("index.json")
    assert index is not None and (index.parent / "00_face.png").exists(), \
        "No earlier gallery attached: set BUILD_GALLERY = True"
    shutil.copytree(index.parent, GALLERY_DIR, dirs_exist_ok=True)
    print("copied the gallery from", index.parent)

gallery = load_gallery()
display(pd.DataFrame([{**e["truth"], "detector": e["label"], "P(fake)": round(e["prob_fake"], 3),
                       "correct": (e["label"] == "fake") == (e["truth"]["label"] == "fake"),
                       "answers stored": ", ".join(MODELS[m] for m in e["answers"])} for e in gallery]))

## Launch
Prints a temporary public `gradio.live` link (valid while this cell runs). Stop the cell to shut the demo down.

In [ ]:
!python app.py --share --checkpoint "$CKPT"